# MedLens AI — Chest X-Ray Model Training

Run this on **Kaggle** (free GPU: Settings > Accelerator > GPU T4 x2) or **Google Colab**.

Dataset: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

On Kaggle: click 'Add Data' and search for the dataset above — no download needed, it mounts directly.

**Output:** `chest_xray_vit.pt` — copy this into `backend/weights/` when done.

In [ ]:
!pip install -q timm torch torchvision

import torch
import torch.nn as nn
import timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# On Kaggle, the dataset mounts at this path once added via 'Add Data'.
# On Colab, upload/unzip the dataset and adjust this path.
DATA_DIR = '/kaggle/input/chest-xray-pneumonia/chest_xray'

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_transform)
test_dataset = datasets.ImageFolder(f'{DATA_DIR}/test', transform=val_transform)

print('Classes:', train_dataset.classes)  # should be ['NORMAL', 'PNEUMONIA']
print('Train size:', len(train_dataset), '| Val size:', len(val_dataset), '| Test size:', len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Fine-tune a ViT-Base/16 (same family you used for your deepfake project)
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
EPOCHS = 8  # keep small — free GPU sessions are time-limited
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step()
    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'chest_xray_vit.pt')
        print('  -> saved new best checkpoint')

In [ ]:
# Final test set evaluation
model.load_state_dict(torch.load('chest_xray_vit.pt'))
test_loss, test_acc = evaluate(model, test_loader)
print(f'Test accuracy: {test_acc:.4f}')

# Download chest_xray_vit.pt from the Kaggle/Colab output panel,
# then place it in backend/weights/chest_xray_vit.pt